In [1]:
#环境与音频检查
import sys
from pathlib import Path

import torch
import transformers

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.audio_utils import(
    load_audio,
    to_mono,
    resample_audio
)

wav_path = root / "results" / "w06d7_resampled_16k.wav"

#读取音频
waveform, sr = load_audio(wav_path)

#统一单声道与采样率
waveform = to_mono(waveform)
waveform = resample_audio(waveform, sr, 16000)

sr = 16000

print("Transformers版本:", transformers.__version__)
print("PyTorch版本:", torch.__version__)

print("\n音频信息")
print("Waveform shape:", tuple(waveform.shape))
print("Sampling rate:", sr)
print("Duration:", waveform.shape[-1] / sr)
print("dtype:", waveform.dtype)

assert waveform.ndim == 2
assert waveform.shape[0] == 1
assert sr == 16000

2026-11-03 04:43:58,075 - INFO - Loaded D:\Projects\speech-ai-learning\results\w06d7_resampled_16k.wav,shape=torch.Size([1, 101376]), sr=16000


Transformers版本: 5.21.0
PyTorch版本: 2.14.1+cu132

音频信息
Waveform shape: (1, 101376)
Sampling rate: 16000
Duration: 6.336
dtype: torch.float32


In [2]:
#执行Processor
from transformers import WhisperProcessor

checkpoint = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(checkpoint)

#(1, T) -> (T,)
audio_np = (
    waveform
    .squeeze(0)
    .detach()
    .cpu()
    .numpy()
)

#Waveform -> Whisper输入特征
inputs = processor(
    audio_np,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features

print("Checkpoint:", checkpoint)
print("Processor类型:",type(processor).__name__)
print("Processor字段:",list(inputs.keys()))

print("\n输入Waveform")
print("shape:", audio_np.shape)
print("dtype:", audio_np.dtype)

print("\n输出input_features")
print("shape:", tuple(input_features.shape))
print("dtype:", input_features.dtype)
print("device:", input_features.device)

assert input_features.ndim == 3
assert input_features.shape[0] == 1
assert input_features.shape[1] == 80
assert input_features.shape[2] == 3000
assert torch.isfinite(input_features).all()

print("\nProcessor检查通过")

2026-11-03 04:50:39,603 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-03 04:50:39,904 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/revision/main "HTTP/1.1 200 OK"
2026-11-03 04:50:40,200 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-03 04:50:40,564 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-03 04:50:40,854 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Checkpoint: openai/whisper-small
Processor类型: WhisperProcessor
Processor字段: ['input_features']

输入Waveform
shape: (101376,)
dtype: float32

输出input_features
shape: (1, 80, 3000)
dtype: torch.float32
device: cpu

Processor检查通过


In [3]:
#检查Processor配置
fe = processor.feature_extractor

print("sampling_rate:", fe.sampling_rate)
print("feature_size:", fe.feature_size)
print("n_fft:", fe.n_fft)
print("hop_length:", fe.hop_length)
print("n_samples:", fe.n_samples)
print("nb_max_frames:", fe.nb_max_frames)

print("\nTokenizer类型:")
print(type(processor.tokenizer).__name__)

sampling_rate: 16000
feature_size: 80
n_fft: 400
hop_length: 160
n_samples: 480000
nb_max_frames: 3000

Tokenizer类型:
WhisperTokenizer


In [4]:
#加载Whisper-small
import torch

from transformers import WhisperForConditionalGeneration

checkpoint = "openai/whisper-small"

# 优先使用CUDA；不可用时使用CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("运行设备:", device)

# 加载模型
model = WhisperForConditionalGeneration.from_pretrained(
    checkpoint
)

model = model.to(device)
model.eval()

print("\n模型加载成功")
print("模型类型:", type(model).__name__)
print("模型checkpoint:", checkpoint)

print("Encoder层数:", model.config.encoder_layers)
print("Decoder层数:", model.config.decoder_layers)
print("模型隐藏维度:", model.config.d_model)
print("词表大小:", model.config.vocab_size)

# 检查模型与输入特征
print("\n输入特征shape:", tuple(input_features.shape))
print("输入特征device:", input_features.device)

print("模型参数device:", next(model.parameters()).device)

运行设备: cuda


2026-11-03 05:16:51,935 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/revision/main "HTTP/1.1 200 OK"
2026-11-03 05:16:52,530 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/973afd24965f72e36ca33b3055d56a652f456b4d/model.safetensors "HTTP/1.1 302 Found"
2026-11-03 05:16:52,530 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
d:\anaconda3\envs\speech-ai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fallwind\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https:/

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

d:\anaconda3\envs\speech-ai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fallwind\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

2026-11-03 05:19:37,318 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-11-03 05:19:37,534 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json?%2Fopenai%2Fwhisper-small%2Fresolve%2F973afd24965f72e36ca33b3055d56a652f456b4d%2Fgeneration_config.json=&etag=%22d3823179a367fa912e3b911b3d2e71ab2a028290%22 "HTTP/1.1 200 OK"
2026-11-03 05:19:37,777 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/openai/whisper-small/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json?%2Fopenai%2Fwhisper-small%2Fresolve%2F973afd24965f72e36ca33b3055d56a652f456b4d%2Fgeneration_config.json=&etag=%22d3823179a367fa912e3b911b3d2e71ab2a028290%22 "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]


模型加载成功
模型类型: WhisperForConditionalGeneration
模型checkpoint: openai/whisper-small
Encoder层数: 12
Decoder层数: 12
模型隐藏维度: 768
词表大小: 51865

输入特征shape: (1, 80, 3000)
输入特征device: cpu
模型参数device: cuda:0


In [5]:
#实验2：调用generate()进行识别
#保证模型输入与模型处于同意设备
features = input_features.to(device)

print("送入generate()的特征:")
print("shape:",tuple(features.shape))
print("dtype:",features.dtype)
print("device",features.device)

#推理：Log-Mel -> Token IDs
with torch.inference_mode():
    generated_ids = model.generate(
        input_features=features,
        language="zh",
        task="transcribe",
        return_timestamps=False,
        max_new_tokens=96,
        do_sample=False
    )

print("\ngenerate()输出:")
print("类型:",type(generated_ids))
print("shape:", tuple(generated_ids.shape))
print("dtype:", generated_ids.dtype)
print("device:", generated_ids.device)

print("\n生成的Token IDs:")
print(generated_ids[0].tolist())

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


送入generate()的特征:
shape: (1, 80, 3000)
dtype: torch.float32
device cuda:0


[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



generate()输出:
类型: <class 'torch.Tensor'>
shape: (1, 30)
dtype: torch.int64
device: cuda:0

生成的Token IDs:
[3582, 50, 4983, 1546, 13557, 4275, 8204, 253, 1486, 115, 41828, 9175, 32063, 26020, 3919, 95, 25703, 9175, 16086, 3316, 238, 19550, 3582, 39823, 19117, 43772, 235, 37960, 7360, 226]


In [6]:
#实验3：将Token IDs解码成文本
#将Tensor转移到CPU，便于打印
ids_cpu = generated_ids.detach().cpu()

print("Token IDs:")
print(ids_cpu[0].tolist())

#1.解码，保留特殊token
text_with_special = processor.batch_decode(
    ids_cpu,
    skip_special_tokens=False
)

#2.解码，跳过特殊token
text_without_special = processor.batch_decode(
    ids_cpu,
    skip_special_tokens=True
)

print("\n包含特殊token的解码结果:")
print(repr(text_with_special[0]))

print("\n去除特殊token的解码结果:")
print(repr(text_without_special[0]))

# 3. 检查Tokenizer
print("\nTokenizer类型:")
print(type(processor.tokenizer).__name__)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Token IDs:
[3582, 50, 4983, 1546, 13557, 4275, 8204, 253, 1486, 115, 41828, 9175, 32063, 26020, 3919, 95, 25703, 9175, 16086, 3316, 238, 19550, 3582, 39823, 19117, 43772, 235, 37960, 7360, 226]

包含特殊token的解码结果:
'大SHA的主要租户包括美林集团及美国运通大型金融机构'

去除特殊token的解码结果:
'大SHA的主要租户包括美林集团及美国运通大型金融机构'

Tokenizer类型:
WhisperTokenizer
